# 🚀 Qwen 2.5 en Google Colab con vLLM & Cloudflare Tunnel

Este notebook levanta un servidor de inferencia de alto rendimiento con **vLLM** para la familia de modelos **Qwen 2.5** (tanto modelos generales como **Qwen2.5-Coder**).

### 🌟 Características:
- ⚡ **vLLM PagedAttention**: Inferencia ultra rápida y gestión de memoria eficiente.
- 🌐 **Cloudflare Tunnel (gratuito, sin registro)**: Expone una URL pública HTTPS (`https://*.trycloudflare.com/v1`).
- 🔌 **Compatible con OpenAI API**: Conecta este endpoint desde tu Mac/PC a Cursor, Continue, Claude Dev, LibreChat, Open WebUI o scripts de Python.
- 💬 **Chat Web UI integrado (Gradio)**: Prueba el modelo directamente en el navegador.

## 1. Verificar GPU y VRAM Asignada
Asegúrate de tener un entorno GPU activado en **Entorno de ejecución > Cambiar tipo de entorno de ejecución** (T4, L4 o A100).

In [ ]:
#@title 🔍 Comprobar GPU
!nvidia-smi
import torch
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"\n✅ GPU Activa: {device_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("\n❌ No se detectó GPU. Activa GPU en Entorno de ejecución > Cambiar tipo de entorno.")

## 2. Configuración del Modelo
### 📊 Guía de compatibilidad de GPUs en Colab Pro:
- **T4 (16 GB)**: `Qwen/Qwen2.5-7B-Instruct`, `Qwen/Qwen2.5-Coder-7B-Instruct` o `Qwen/Qwen2.5-14B-Instruct-AWQ` (con contexto 4096).
- **L4 (24 GB)**: `Qwen/Qwen2.5-7B-Instruct`, `Qwen/Qwen2.5-14B-Instruct-AWQ`, `Qwen/Qwen2.5-Coder-14B-Instruct-AWQ`.
- **A100 (40 GB / 80 GB)**: `Qwen/Qwen2.5-14B-Instruct`, `Qwen/Qwen2.5-32B-Instruct-AWQ`, `Qwen/Qwen2.5-Coder-32B-Instruct-AWQ` o incluso `Qwen/Qwen2.5-72B-Instruct-AWQ` (en 80GB).

In [ ]:
#@title ⚙️ Parámetros de Inferencia
MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct" #@param ["Qwen/Qwen2.5-Coder-7B-Instruct", "Qwen/Qwen2.5-7B-Instruct", "huihui-ai/Huihui-Qwen3.8-27B-abliterated", "Qwen/Qwen2.5-14B-Instruct-AWQ", "Qwen/Qwen2.5-Coder-14B-Instruct-AWQ", "Qwen/Qwen2.5-32B-Instruct-AWQ", "Qwen/Qwen2.5-Coder-32B-Instruct-AWQ", "personalizado"] {allow-input: true}
CUSTOM_MODEL_ID = "" #@param {type:"string"}
MAX_MODEL_LEN = 8192 #@param [2048, 4096, 8192, 16384, 32768] {type:"raw"}
GPU_UTILIZATION = 0.90 #@param {type:"slider", min:0.70, max:0.95, step:0.05}
API_KEY = "colab-qwen-secret" #@param {type:"string"}
PORT = 8000

if MODEL_ID == "personalizado" and CUSTOM_MODEL_ID.strip():
    MODEL_ID = CUSTOM_MODEL_ID.strip()

print(f"🎯 Modelo a cargar: {MODEL_ID}")
print(f"📏 Context Length: {MAX_MODEL_LEN}")
print(f"🔑 API Key: {API_KEY}")

## 3. Instalar Dependencias
Instalamos `vllm`, paquetes cliente (`openai`, `gradio`) y el binario de `cloudflared` para túnel seguro sin registrarse.

In [ ]:
#@title 📦 Instalar vLLM y Cloudflared
!pip install -q vllm openai gradio

# Descargar e instalar Cloudflared para el túnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!rm -f cloudflared-linux-amd64.deb
print("✅ Instalación completada.")

## 4. Iniciar Servidor vLLM y Generar URL Pública

In [ ]:
#@title 🚀 Arrancar Servidor vLLM y Cloudflare Tunnel
import subprocess
import time
import requests
import re
import os

# Matar procesos previos si existen
!pkill -f vllm > /dev/null 2>&1
!pkill -f cloudflared > /dev/null 2>&1
time.sleep(1)

# Detectar si requiere flag de cuantización automática
quant_flag = ""
if "AWQ" in MODEL_ID.upper():
    quant_flag = "--quantization awq"
elif "FP8" in MODEL_ID.upper():
    quant_flag = "--quantization fp8"

vllm_cmd = (
    f"python3 -m vllm.entrypoints.openai.api_server "
    f"--model {MODEL_ID} "
    f"--port {PORT} "
    f"--gpu-memory-utilization {GPU_UTILIZATION} "
    f"--max-model-len {MAX_MODEL_LEN} "
    f"--api-key {API_KEY} "
    f"{quant_flag} --trust-remote-code"
)

print(f"⏳ Comando: {vllm_cmd}\n")
print("⏳ Iniciando servidor vLLM (descargando pesos y reservando VRAM, ~1-3 minutos)...\n")
with open("vllm.log", "w") as vllm_log:
    vllm_proc = subprocess.Popen(vllm_cmd, shell=True, stdout=vllm_log, stderr=vllm_log)

# Iniciar Cloudflare Tunnel
tunnel_cmd = f"cloudflared tunnel --url http://localhost:{PORT}"
with open("tunnel.log", "w") as tunnel_log:
    tunnel_proc = subprocess.Popen(tunnel_cmd, shell=True, stdout=tunnel_log, stderr=tunnel_log)

# Obtener URL pública de Cloudflare
public_url = None
for _ in range(30):
    time.sleep(2)
    try:
        with open("tunnel.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break
    except Exception:
        pass

if public_url:
    print("=" * 65)
    print("🎉 ¡TÚNEL PÚBLICO ACTIVADO CON ÉXITO!")
    print(f"🌐 Base URL para tus apps / Cursor: {public_url}/v1")
    print(f"🔑 API Key: {API_KEY}")
    print(f"🤖 Model ID: {MODEL_ID}")
    print("=" * 65 + "\n")
else:
    print("⚠️ No se encontró la URL del túnel todavía. Revisa tunnel.log")

# Esperar a que el servidor de vLLM responda en /health
print("⏳ Esperando que vLLM termine de inicializar el modelo:")
ready = False
for i in range(80):
    try:
        r = requests.get(f"http://localhost:{PORT}/health", timeout=2)
        if r.status_code == 200:
            ready = True
            break
    except Exception:
        pass
    time.sleep(4)
    print(".", end="", flush=True)

if ready:
    print("\n\n🚀 ¡Servidor vLLM LISTO Y OPERATIVO!")
else:
    print("\n\n❌ Hubo un retraso o error al levantar vLLM. Últimas líneas de vllm.log:")
    !tail -n 25 vllm.log

## 5. Prueba de Inferencia (Python OpenAI Client)

In [ ]:
#@title 🧪 Prueba Rápida de Streaming
from openai import OpenAI

client = OpenAI(
    base_url=f"http://localhost:{PORT}/v1",
    api_key=API_KEY
)

prompt = "Explica brevemente qué es la inferencia con vLLM y por qué es rápida."

print(f"Pregunta: {prompt}\n")
stream = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": "Eres un asistente experto en Inteligencia Artificial."},
        {"role": "user", "content": prompt}
    ],
    temperature=0.7,
    max_tokens=512,
    stream=True
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

## 6. (Opcional) Web Chat UI Interactivo con Gradio
Lanza un chat visual interactivo para probar el modelo directamente en el notebook o con enlace público compartible.

In [ ]:
#@title 💬 Iniciar Chat Gradio
import gradio as gr
from openai import OpenAI

local_client = OpenAI(base_url=f"http://localhost:{PORT}/v1", api_key=API_KEY)

def chat_stream(message, history):
    messages = [{"role": "system", "content": "Eres un asistente inteligente y útil basado en Qwen."}]
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})
    messages.append({"role": "user", "content": message})
    
    response = local_client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        temperature=0.7,
        stream=True
    )
    
    bot_message = ""
    for chunk in response:
        delta = chunk.choices[0].delta.content or ""
        bot_message += delta
        yield bot_message

demo = gr.ChatInterface(
    fn=chat_stream,
    title=f"🤖 {MODEL_ID}",
    description="Inferencia en Colab Pro con vLLM y túnel Cloudflare",
    examples=[
        "Escribe una función en Python para ordenar una lista de diccionarios por clave.",
        "Explica la diferencia entre Fine-Tuning y RAG."
    ]
)
demo.launch(share=True, inline=True)

## 7. Monitorización de Recursos y Logs

In [ ]:
#@title 📊 Ver estado de GPU y últimos logs de vLLM
!nvidia-smi
print("\n--- Últimas 20 líneas de vllm.log ---")
!tail -n 20 vllm.log